# Conda Environment: 07_rTPC
---

# Library Imports

In [2]:
library('rTPC')
library('purrr')
library('nls.multstart')
library('broom')
library('dplyr')
library('ggplot2')
library('qpcR')
library('MuMIn');

---

# Create Output Directories

In [2]:
system('mkdir -p ../Datasets/Optimum_temperature')

---

# Import Data

## Sample Metadata

In [3]:
sample_data = read.csv("../Datasets/sample-data.csv")
sample_data = sample_data[c('Sample','Temperature')]

## Absolute abundance sequence table

In [4]:
data = read.csv("../Datasets/Absolute_abundance/StageOneActive_AA_long.csv")
data$Relative_abundance = NULL

In [5]:
data = merge(data, sample_data, by= 'Sample',all.y= FALSE, sort= FALSE)

## Combined fold-change and niche breadth data

In [6]:
# Load in ASV data from niche breadth
asvs <- read.csv("../Datasets/Niche_breadth//StageOneActive_NicheBreadth.csv")

# Filter to ASVs above limit of quantitation
asvs = asvs %>% filter(Below.LOQ == 'N')

# Restore ASV_ID column
asvs$ASV_ID = asvs$X
asvs$X = NULL

## Limit absolute abundance sequence table to taxa with niche breadth

In [7]:
data = data %>% filter(ASV_ID %in% asvs$ASV_ID)
stopifnot(length(unique(data$ASV_ID)) == length(unique(asvs$ASV_ID)))

---

# Calculate Optimum Temperature for All Taxa

## Fitting all models with optimum temperature

In [10]:
find_opt <- function(dataset, taxon) {

# Lists to store results
models = c()
aic_values = c()
optimum_estimates = c()
    
# Subset to data of interest
data = filter(dataset, ASV_ID == taxon)

# Set abundance at 0 degrees to 0    
z1 <- data.frame(Sample= 'Z1', ASV_ID= taxon, Absolute_abundance= 0, Temperature= 0)
z2 <- data.frame(Sample= 'Z2', ASV_ID= taxon, Absolute_abundance= 0, Temperature= 0)
z3 <- data.frame(Sample= 'Z3', ASV_ID= taxon, Absolute_abundance= 0, Temperature= 0)
data <- rbind(data, z1)
data <- rbind(data, z2)
data <- rbind(data, z3)


######################################
### Try fitting Deutsch 2008 START ###
######################################

# Choose model(s) to use
model = 'deutsch_2008'

# Get starting values
deutsch_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
deutsch_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
deutsch_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_deutsch <- nls_multstart(Absolute_abundance ~ Deutsch_2008(temp = Temperature, rmax, topt, ctmax, a),
                     data = data,
                     iter =  10000,
                     start_lower = deutsch_starting_values - 10,
                     start_upper = deutsch_starting_values + 10,
                     lower = deutsch_lower_limits,
                     upper = deutsch_upper_limits,
                     supp_errors = 'Y')

# Get optimum temperature    
optimum = calc_params(fit_deutsch) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_deutsch)){aic_value = NA} else {aic_value = AICc(fit_deutsch)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    

####################################
### Try fitting Deutsch 2008 END ###
####################################

#######################################
### Try fitting Gaussian 1987 START ###
#######################################

# Choose model(s) to use
model = 'gaussian_1987'

# Get starting values
gaussian_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
gaussian_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
gaussian_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_gaussian <- nls_multstart(Absolute_abundance ~ gaussian_1987(temp = Temperature, rmax, topt, a),
                     data = data,
                     iter =  10000,
                     start_lower = gaussian_starting_values - 10,
                     start_upper = gaussian_starting_values + 10,
                     lower = gaussian_lower_limits,
                     upper = gaussian_upper_limits,
                     supp_errors = 'Y')
    

# Get optimum temperature    
optimum = calc_params(fit_gaussian) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_gaussian)){aic_value = NA} else {aic_value = AICc(fit_gaussian)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)

    
#####################################
### Try fitting Gaussian 1987 END ###
#####################################

#####################################
### Try fitting Joehnk 2008 START ###
#####################################

# Choose model(s) to use
model = 'joehnk_2008'

# Get starting values
joehnk_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
joehnk_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
joehnk_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_joehnk <- nls_multstart(Absolute_abundance ~ joehnk_2008(temp = Temperature,  rmax, topt, a, b, c),
                     data = data,
                     iter =  10000,
                     start_lower = joehnk_starting_values - 10,
                     start_upper = joehnk_starting_values + 10,
                     lower = joehnk_lower_limits,
                     upper = joehnk_upper_limits,
                     supp_errors = 'Y')


# Get optimum temperature    
optimum = calc_params(fit_joehnk) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_joehnk)){aic_value = NA} else {aic_value = AICc(fit_joehnk)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    

###################################
### Try fitting Joehnk 2008 END ###
###################################

############################################
### Try fitting Johnson Lewin 1946 START ###
############################################

# Choose model(s) to use
model = 'johnsonlewin_1946'

# Get starting values
jl_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
jl_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
jl_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_jl <- nls_multstart(Absolute_abundance ~ johnsonlewin_1946(temp = Temperature,  r0, e, eh, topt),
                     data = data,
                     iter =  10000,
                     start_lower = jl_starting_values - 10,
                     start_upper = jl_starting_values + 10,
                     lower = jl_lower_limits,
                     upper = jl_upper_limits,
                     supp_errors = 'Y')


# Get optimum temperature    
optimum = calc_params(fit_jl) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_jl)){aic_value = NA} else {aic_value = AICc(fit_jl)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)

##########################################
### Try fitting Johnson Lewin 1946 END ###
##########################################

                             
####################################################
### Try fitting Lobry-Rosso-Flandros 1991 START ###
####################################################

# Choose model(s) to use
model = 'lrf_1991'

# Get starting values
lrf_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
lrf_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
lrf_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_lrf <- nls_multstart(Absolute_abundance ~ lrf_1991(temp = Temperature, rmax, topt, tmin, tmax),
                     data = data,
                     iter =  10000,
                     start_lower = lrf_starting_values - 10,
                     start_upper = lrf_starting_values + 10,
                     lower = lrf_lower_limits,
                     upper = lrf_upper_limits,
                     supp_errors = 'Y')
    

# Get optimum temperature    
optimum = calc_params(fit_lrf) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_lrf)){aic_value = NA} else {aic_value = AICc(fit_lrf)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    
#################################################
### Try fitting Lobry-Rosso-Flandros 1991 END ###
#################################################

                             
################################################
### Try fitting modified Gaussian 2006 START ###
################################################

# Choose model(s) to use
model = 'modifiedgaussian_2006'

# Get starting values
modgaussian_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
modgaussian_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
modgaussian_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_modgaussian <- nls_multstart(Absolute_abundance ~ modifiedgaussian_2006(temp = Temperature, rmax, topt, a, b),
                     data = data,
                     iter =  10000,
                     start_lower = modgaussian_starting_values - 10,
                     start_upper = modgaussian_starting_values + 10,
                     lower = modgaussian_lower_limits,
                     upper = modgaussian_upper_limits,
                     supp_errors = 'Y')


# Get optimum temperature    
optimum = calc_params(fit_modgaussian) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_modgaussian)){aic_value = NA} else {aic_value = AICc(fit_modgaussian)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)

##############################################
### Try fitting modified Gaussian 2006 END ###
##############################################
                             
                             
######################################
### Try fitting O'Neill 1972 START ###
######################################

# Choose model(s) to use
model = 'oneill_1972'

# Get starting values
oneill_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
oneill_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
oneill_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_oneill <- nls_multstart(Absolute_abundance ~ oneill_1972(temp = Temperature, rmax, ctmax, topt, q10),
                     data = data,
                     iter =  10000,
                     start_lower = oneill_starting_values - 10,
                     start_upper = oneill_starting_values + 10,
                     lower = oneill_lower_limits,
                     upper = oneill_upper_limits,
                     supp_errors = 'Y')
    

# Get optimum temperature    
optimum = calc_params(fit_oneill) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_oneill)){aic_value = NA} else {aic_value = AICc(fit_oneill)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    
####################################
### Try fitting O'Neill 1972 END ###
####################################


######################################
### Try fitting Rezende 2019 START ###
######################################

# Choose model(s) to use
model = 'rezende_2019'
    
# Get starting values
rezende_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
rezende_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
rezende_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_rezende <- nls_multstart(Absolute_abundance ~ rezende_2019(temp = Temperature, q10, a, b, c),
                     data = data,
                     iter =  10000,
                     start_lower = rezende_starting_values - 10,
                     start_upper = rezende_starting_values + 10,
                     lower = rezende_lower_limits,
                     upper = rezende_upper_limits,
                     supp_errors = 'Y')

# Get optimum temperature    
optimum = calc_params(fit_rezende) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_rezende)){aic_value = NA} else {aic_value = AICc(fit_rezende)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    

####################################
### Try fitting Rezende 2019 END ###
####################################

                             
####################################
### Try fitting Spain 1982 START ###
####################################

# Choose model(s) to use
model = 'spain_1982'

# Get starting values
spain_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
spain_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
spain_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_spain <- nls_multstart(Absolute_abundance ~ spain_1982(temp = Temperature, a, b, c, r0),
                     data = data,
                     iter =  10000,
                     start_lower = spain_starting_values - 10,
                     start_upper = spain_starting_values + 10,
                     lower = spain_lower_limits,
                     upper = spain_upper_limits,
                     supp_errors = 'Y')

# Get optimum temperature    
optimum = calc_params(fit_spain) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_spain)){aic_value = NA} else {aic_value = AICc(fit_spain)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    

##################################
### Try fitting Spain 1982 END ###
##################################

                             
######################################
### Try fitting Weibull 1995 START ###
######################################

# Choose model(s) to use
model = 'weibull_1995'

# Get starting values
weibull_starting_values = get_start_vals(data$Temperature, data$Absolute_abundance, model_name = model)

# Get limits
weibull_lower_limits <- get_lower_lims(data$Temperature, data$Absolute_abundance, model_name = model)
weibull_upper_limits <- get_upper_lims(data$Temperature, data$Absolute_abundance, model_name = model)

# fit model
fit_weibull <- nls_multstart(Absolute_abundance ~ weibull_1995(temp = Temperature,  a, topt, b, c),
                     data = data,
                     iter =  10000,
                     start_lower = weibull_starting_values - 10,
                     start_upper = weibull_starting_values + 10,
                     lower = weibull_lower_limits,
                     upper = weibull_upper_limits,
                     supp_errors = 'Y')

# Get optimum temperature    
optimum = calc_params(fit_weibull) %>% mutate_all(round, 2)
optimum = optimum$topt

# Calculate AICc
if (is.null(fit_weibull)){aic_value = NA} else {aic_value = AICc(fit_weibull)}

# Update list
models = c(models, model)
aic_values = c(aic_values, aic_value)
optimum_estimates = c(optimum_estimates, optimum)
    
####################################
### Try fitting Weibull 1995 END ###
####################################


# Make dataframe of results
results = data.frame(Model= models, Optimum= optimum_estimates, AICc= aic_values)
results = na.omit(results)
results$Weights = akaike.weights(results$AICc)$weights

# Find lowest AICc and calculate delta
lowest_aic = min(results$AICc)
results$Delta = results$AICc - lowest_aic

# Remove models where delta AICc > 4
results = results %>% filter(Delta < 4)

# Weighted averaging of Topt
results$Weighted_topt = results$Optimum * results$Weights
optimum = sum(results$Weighted_topt)
    
return(round(optimum,2))
#return(results)
}

In [13]:
# Set random seed
set.seed(11)


# Estimate optimum temperature for all ASVs
optima = c()


# Warning: takes about 10.5 hours to run
for (taxon in unique(data$ASV_ID)){
    optima <- append(optima,find_opt(dataset= data, taxon= taxon))
    }

In [14]:
# Create output dataframe
asv_optima = data.frame(ASV_ID= unique(data$ASV_ID), Optimum= optima)

# Export to file
write.csv(asv_optima,'../Datasets/Optimum_temperature/Optima.csv', row.names= FALSE)